# Complementary categories

**What this notebook does.** It applies the support and lift thresholds to the
scored co-purchase pairs and writes the final complementary-category table to
`data/complementary_pairs.pkl`. Each row is a directed pair — one category path
buying into another — with the co-purchase count, both marginals, and the two
metrics behind it.

**What it is used for.** That table is the complementary-category signal for
the recommender: given the category of an item a user is looking at, it says
which categories are worth recommending alongside it, and how much co-purchase
evidence stands behind each suggestion.

**Where the numbers come from.** The scored pairs are built by
`complementary_category_analysis.ipynb`, which reads the raw sources — Amazon's
`also_buy` lists, resolved to a category path on both ends — scores every pair
by support and lift, and saves the lot to `data/pair_stats.pkl`. The charts in
that notebook are how `MIN_EDGES` and `MIN_LIFT` below were chosen. **Run it
first**; this notebook only reads its output, so re-running at a different
threshold costs seconds rather than minutes.

Every function lives in `complementary_categories.py` — this notebook defines
no logic of its own:

```python
pair_stats          = pd.read_pickle(PAIR_STATS_PATH)          # from the analysis notebook
complementary_pairs = filter_pairs(pair_stats, MIN_EDGES, MIN_LIFT)
save_pairs(complementary_pairs, OUT_PATH)
```

In [5]:
import sys
from pathlib import Path

# Make the package importable when running from this folder
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from complementary_categories import DST_COLS, SRC_COLS, filter_pairs, save_pairs

DATA_DIR = PROJECT_ROOT / "data"
PAIR_STATS_PATH = DATA_DIR / "pair_stats.pkl"
OUT_PATH = DATA_DIR / "complementary_pairs.pkl"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the scored pairs

`pair_stats.pkl` is every distinct `(source path -> target path)` pair the
`also_buy` edges trace out, unfiltered, with `edges` (the co-purchase count),
the two marginals, `support` and `lift`. `N`, the total resolved edges the two
metrics are shares of, is recoverable from the table itself: every resolved
edge lands in exactly one pair, so it is `edges.sum()`.

In [6]:
if not PAIR_STATS_PATH.exists():
    raise FileNotFoundError(
        f"{PAIR_STATS_PATH} not found. Run complementary_category_analysis.ipynb "
        "first — it builds the pairs from the raw sources, scores them, and saves "
        "this table."
    )

pair_stats = pd.read_pickle(PAIR_STATS_PATH)
n_edges = int(pair_stats["edges"].sum())

print(f"distinct pairs : {len(pair_stats):,}")
print(f"resolved edges : {n_edges:,}")
print(f"support        : {pair_stats['support'].min():.3g} .. {pair_stats['support'].max():.3g}")
print(f"lift           : {pair_stats['lift'].min():,.2f} .. {pair_stats['lift'].max():,.0f}")

with pd.option_context("display.float_format", lambda x: f"{x:,.6g}"):
    display(pair_stats.head(5))

distinct pairs : 30,392
resolved edges : 1,026,873
support        : 9.74e-07 .. 0.039
lift           : 0.00 .. 1,026,873


,src_cat_2,src_cat_3,src_cat_4,dst_cat_2,dst_cat_3,dst_cat_4,edges,src_edges,dst_edges,support,lift
0,Home Dcor,Home Dcor Accents,Ornaments,Home Dcor,Home Dcor Accents,Ornaments,40095,48426,44936,0.0390457,18.9206
1,Wall Art,Posters & Prints,Missing,Wall Art,Posters & Prints,Missing,38813,47710,45745,0.0377973,18.2617
2,Home Dcor,Home Dcor Accents,Collectible Figurines,Home Dcor,Home Dcor Accents,Collectible Figurines,34984,45771,46396,0.0340685,16.9167
3,Kitchen & Dining,Bakeware,Baking Tools & Accessories,Kitchen & Dining,Bakeware,Baking Tools & Accessories,31348,47962,43909,0.0305276,15.2854
4,Home Dcor,Home Dcor Accents,Decorative Accessories,Home Dcor,Home Dcor Accents,Decorative Accessories,27188,40206,41919,0.0264765,16.565


## 2. The thresholds

Read off the survival curves in `complementary_category_analysis.ipynb` §6.
Change them here and re-run — nothing above this cell depends on them.

`MIN_EDGES` is a whole number of co-purchases rather than a support fraction
because it is the readable unit; the equivalent support floor is
`MIN_EDGES / n_edges`, printed in §3.

Neither threshold works alone. Support on its own promotes whatever is popular,
since a high-traffic category pairs with everything. Lift on its own does the
opposite: a single-edge pair between two otherwise-unseen paths scores `N`, the
maximum possible, on one co-purchase.

In [7]:
MIN_EDGES = 5     # support floor, as a whole number of co-purchases
MIN_LIFT = 2.0    # at least twice as often as independence would predict

## 3. Apply the split and save

Each threshold's own count prints before the combined one, which is the
quickest way to see which of the two is doing the cutting.

Pairs whose source and target paths are identical are kept: a chair listed with
another chair is a real `also_buy` edge. Drop them here if the downstream use
needs strict complements.

`save_pairs` casts the six category columns to `category` dtype before
pickling, which makes the file about a quarter the size of the equivalent CSV —
a few hundred distinct strings repeat across thousands of rows. Re-running this
cell overwrites the output.

In [8]:
complementary_pairs = filter_pairs(pair_stats, MIN_EDGES, MIN_LIFT)
save_pairs(complementary_pairs, OUT_PATH)

by_support = pair_stats["edges"] >= MIN_EDGES
by_lift = pair_stats["lift"] >= MIN_LIFT
n_pairs = len(pair_stats)
kept_self = (complementary_pairs[SRC_COLS].to_numpy()
             == complementary_pairs[DST_COLS].to_numpy()).all(axis=1)

print(f"thresholds         : edges >= {MIN_EDGES} (support >= {MIN_EDGES / n_edges:.3g}) | "
      f"lift >= {MIN_LIFT:g}")
print(f"support alone      : {by_support.sum():,} of {n_pairs:,} pairs ({by_support.mean():.1%})")
print(f"lift alone         : {by_lift.sum():,} of {n_pairs:,} pairs ({by_lift.mean():.1%})")
print(f"both               : {len(complementary_pairs):,} of {n_pairs:,} pairs "
      f"({len(complementary_pairs) / n_pairs:.1%})")
print(f"edges covered      : {complementary_pairs['edges'].sum():,} of {n_edges:,} "
      f"({complementary_pairs['edges'].sum() / n_edges:.1%})")
print(f"source paths       : {complementary_pairs[SRC_COLS].drop_duplicates().shape[0]:,}")
print(f"target paths       : {complementary_pairs[DST_COLS].drop_duplicates().shape[0]:,}")
print(f"same path both ends: {kept_self.sum():,}")
print(f"\nsaved -> {OUT_PATH.relative_to(PROJECT_ROOT)} "
      f"({OUT_PATH.stat().st_size / 1e6:.2f} MB)")

with pd.option_context("display.float_format", lambda x: f"{x:,.6g}"):
    display(complementary_pairs.head(20))

thresholds         : edges >= 5 (support >= 4.87e-06) | lift >= 2
support alone      : 11,199 of 30,392 pairs (36.8%)
lift alone         : 13,295 of 30,392 pairs (43.7%)
both               : 5,845 of 30,392 pairs (19.2%)
edges covered      : 807,578 of 1,026,873 (78.6%)
source paths       : 455
target paths       : 512
same path both ends: 391

saved -> data/complementary_pairs.pkl (0.31 MB)


,src_cat_2,src_cat_3,src_cat_4,dst_cat_2,dst_cat_3,dst_cat_4,edges,src_edges,dst_edges,support,lift
0,Home Dcor,Home Dcor Accents,Ornaments,Home Dcor,Home Dcor Accents,Ornaments,40095,48426,44936,0.0390457,18.9206
1,Wall Art,Posters & Prints,Missing,Wall Art,Posters & Prints,Missing,38813,47710,45745,0.0377973,18.2617
2,Home Dcor,Home Dcor Accents,Collectible Figurines,Home Dcor,Home Dcor Accents,Collectible Figurines,34984,45771,46396,0.0340685,16.9167
3,Kitchen & Dining,Bakeware,Baking Tools & Accessories,Kitchen & Dining,Bakeware,Baking Tools & Accessories,31348,47962,43909,0.0305276,15.2854
4,Home Dcor,Home Dcor Accents,Decorative Accessories,Home Dcor,Home Dcor Accents,Decorative Accessories,27188,40206,41919,0.0264765,16.565
5,Kitchen & Dining,Bakeware,Candy Making Supplies,Kitchen & Dining,Bakeware,Candy Making Supplies,21268,30578,28896,0.0207114,24.717
6,Home Dcor,Candles & Holders,Candles,Home Dcor,Candles & Holders,Candles,18667,25287,25102,0.0181785,30.1985
7,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,17732,30248,26228,0.017268,22.9516
8,Home Dcor,Home Fragrance,Incense & Incense Holders,Home Dcor,Home Fragrance,Incense & Incense Holders,17441,18523,18400,0.0169846,52.5483
9,Kitchen & Dining,Kitchen Utensils & Gadgets,Kitchen Accessories,Kitchen & Dining,Kitchen Utensils & Gadgets,Kitchen Accessories,16285,23370,22200,0.0158588,32.2324


---

Read the result back with:

```python
complementary_pairs = pd.read_pickle(DATA_DIR / "complementary_pairs.pkl")
```

To rebuild everything from the raw sources in one call, skipping both notebooks
and the intermediate table:

```python
from complementary_categories import run_complementary_pairs

complementary_pairs = run_complementary_pairs(
    features=DATA_DIR / "df_features.pkl",
    catalogue_path=DATA_DIR / "meta_Home_and_Kitchen_filtered.csv",
    taxonomy_path=DATA_DIR / "category_taxonomy.json",
    min_edges=5,
    min_lift=2.0,
    out_path=DATA_DIR / "complementary_pairs.pkl",
)
```